# Kaggle Ingestion part

# setup

In [1]:
import sys, platform, os
from subprocess import run

def pip_install(pkgs):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + pkgs
    r = run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError("pip install failed")

packages = [
    "docling",
    "sentence-transformers",
    "faiss-cpu",
    "langchain-text-splitters",
    "tiktoken",
    "orjson",
    "pandas",
    "tqdm",
]

pip_install(packages)

print("Python:", sys.version)
print("Platform:", platform.platform())

# Versions + GPU check
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

import faiss
print("FAISS:", faiss.__version__)

import tiktoken
print("tiktoken:", tiktoken.__version__)

import orjson
print("orjson:", orjson.__version__)

import pandas as pd
print("pandas:", pd.__version__)

from sentence_transformers import SentenceTransformer
print("sentence-transformers:", SentenceTransformer.__module__.split(".")[0])

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
Torch: 2.10.0+cu128
CUDA available: True
CUDA device: Tesla T4
FAISS: 1.13.2
tiktoken: 0.12.0
orjson: 3.11.7
pandas: 2.3.3


2026-03-25 11:40:11.517038: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774438811.752716      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774438811.813846      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774438812.316373      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774438812.316410      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774438812.316413      55 computation_placer.cc:177] computation placer alr

sentence-transformers: sentence_transformers


# Global config + imports

In [2]:
from __future__ import annotations

import re
import json
import math
import time
import gzip
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import orjson
import tiktoken
import faiss
import torch

from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

from openai import OpenAI


# =========================
# Global config
# =========================

# Sections / summaries
# SECTION_SUMMARY_MODEL = "stepfun/step-3.5-flash:free"
SECTION_SUMMARY_MODEL = "openai/gpt-5-nano"

# Для небольших технических PDF ничего не отрезаем
TRIM_FRONT_PAGES = 0
TRIM_BACK_PAGES = 0

# Merge small neighboring sections before summarization
# под документы ~15–30 страниц:
MIN_MERGED_SECTION_CHARS = 1500
MAX_MERGED_SECTION_CHARS = 12000
MIN_MERGED_SECTION_PAGES = 2
MAX_MERGED_SECTION_PAGES = 8

# Parallel summarization
SECTION_SUMMARY_CONCURRENCY = 8

SECTION_RETRIEVAL_TOP_K = 8

# Section splitting / fallback
# Для маленьких документов fallback делаем компактнее
MAX_SECTION_PAGES_FALLBACK = 6
FALLBACK_PAGE_OVERLAP = 1

# Oversized section handling
MAX_SECTION_SUMMARY_INPUT_CHARS = 24000
OVERSIZED_SECTION_CHAR_LIMIT = 32000

# OpenRouter
# OPENROUTER_API_KEY = "sk-or-v1-""

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

PIPELINE_VERSION = "gazprom-techdocs-bge-m3_v2"

# Chunking
CHUNK_SIZE = 300
CHUNK_OVERLAP = 50
SPLITTER_MODEL_NAME = "gpt-4o"  # used for tiktoken encoder in splitter

# Embeddings
EMBEDDING_MODEL = "BAAI/bge-m3"

# Indexing
FAISS_INDEX_TYPE = "IndexFlatIP"  # cosine-like via IP + L2 normalize
PAGE_BASE = 1  # page_no is strictly 1-based everywhere

EMBED_BATCH_SIZE = 256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Output dirs
ARTIFACTS_ROOT = Path("/kaggle/working/artifacts")
DIRS = {
    "parsed_reports": ARTIFACTS_ROOT / "parsed_reports",
    "merged_reports": ARTIFACTS_ROOT / "merged_reports",
    "sectioned_reports": ARTIFACTS_ROOT / "sectioned_reports",
    "chunked_reports": ARTIFACTS_ROOT / "chunked_reports",
    "vector_dbs": ARTIFACTS_ROOT / "vector_dbs",
    "reports": ARTIFACTS_ROOT / "reports",
    "cache": ARTIFACTS_ROOT / "cache",
    "emb_cache": ARTIFACTS_ROOT / "cache" / "embeddings",
    "section_emb_cache": ARTIFACTS_ROOT / "cache" / "section_embeddings",
    "chunk_emb_cache": ARTIFACTS_ROOT / "cache" / "chunk_embeddings",
}

def ensure_dirs(dirs: Dict[str, Path]) -> None:
    for p in dirs.values():
        p.mkdir(parents=True, exist_ok=True)

ensure_dirs(DIRS)


# =========================
# JSON helpers
# =========================

def read_json(path: Path) -> Any:
    with path.open("rb") as f:
        return orjson.loads(f.read())

def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    data = orjson.dumps(obj, option=orjson.OPT_INDENT_2 | orjson.OPT_SORT_KEYS)
    with path.open("wb") as f:
        f.write(data)

def write_jsonl(path: Path, rows: List[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        for r in rows:
            f.write(orjson.dumps(r))
            f.write(b"\n")

def read_jsonl(path: Path) -> List[dict]:
    rows = []
    with path.open("rb") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(orjson.loads(line))
    return rows

# Token encoder used consistently (for splitter + length_tokens)
ENC = tiktoken.encoding_for_model(SPLITTER_MODEL_NAME)

def count_tokens(text: str) -> int:
    return len(ENC.encode(text))

# Splitter
def make_splitter() -> RecursiveCharacterTextSplitter:
    return RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        model_name=SPLITTER_MODEL_NAME,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )

SPLITTER = make_splitter()

embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

EMBED_DIM = int(embedder.get_sentence_embedding_dimension())
print("Embedding dim:", EMBED_DIM)

print("Artifacts root:", ARTIFACTS_ROOT)
print("Device:", DEVICE)
print("Embedding model:", EMBEDDING_MODEL)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding dim: 1024
Artifacts root: /kaggle/working/artifacts
Device: cuda
Embedding model: BAAI/bge-m3


# Data download + paths

Скачиваем датасет через `kagglehub`, определяем пути, создаём выходные папки.

In [3]:

import kagglehub
from pathlib import Path

# Download latest version
path = kagglehub.dataset_download("denismuradyan/gazprom-pdf-rag-index")
print("Path to dataset files:", path)

DATASET_PATH = Path(path)
assert DATASET_PATH.exists(), f"Missing: {DATASET_PATH}"

pdf_paths = sorted(DATASET_PATH.glob("*.pdf"))
assert pdf_paths, f"No PDF files found in {DATASET_PATH}"

print("DATASET_PATH:", DATASET_PATH)
print("Found PDFs:", len(pdf_paths))
for p in pdf_paths[:10]:
    print("-", p.name)

Path to dataset files: /kaggle/input/datasets/denismuradyan/gazprom-pdf-rag-index
DATASET_PATH: /kaggle/input/datasets/denismuradyan/gazprom-pdf-rag-index
Found PDFs: 14
- 2019-09-17-safety-policy.pdf
- 34.201-2020.pdf
- 34.602-2020.pdf
- 4293767636.pdf
- GOST_R_2.105-2019.pdf
- Gost_7-32_2017.pdf
- cto_gazprom_1.13-2012.pdf
- koncept_tr_2023.pdf
- pravila_vklychenija_produkcii_v_mtr.pdf
- rosstandart_290725-gost.pdf


# Load subset + build docs_meta + sanity checks

- Формируем `docs_meta` по `doc_id`
- Проверяем, что для каждого `doc_id` существует `pdfs/{doc_id}.pdf`

In [4]:
docs_meta: Dict[str, Dict[str, Any]] = {}
doc_id_list: List[str] = []

def make_doc_id(pdf_path: Path) -> str:
    return pdf_path.stem

for pdf_path in pdf_paths:
    doc_id = make_doc_id(pdf_path)
    doc_id_list.append(doc_id)
    docs_meta[doc_id] = {
        "doc_id": doc_id,
        "source_file": pdf_path.name,
        "title": pdf_path.stem,
        "pdf_path": str(pdf_path),
    }

doc_id_list = sorted(set(doc_id_list))

print("Documents:", len(doc_id_list))
print("Example doc_id:", doc_id_list[0])
print("Example meta:", docs_meta[doc_id_list[0]])

Documents: 14
Example doc_id: 2019-09-17-safety-policy
Example meta: {'doc_id': '2019-09-17-safety-policy', 'source_file': '2019-09-17-safety-policy.pdf', 'title': '2019-09-17-safety-policy', 'pdf_path': '/kaggle/input/datasets/denismuradyan/gazprom-pdf-rag-index/2019-09-17-safety-policy.pdf'}


# Text normalization: repair + safe cleanup

Здесь фиксируем 2 функции:

- `repair_text(text)` — детерминированная нормализация строго по фиксированным паттернам:
  - slash-команды `/zero...`, `/percent...`, `/lparen...` → символы
  - удаление `glyph<...>`
  - разворот `/A.cap` → `A`

- `safe_cleanup_text(text)` — безопасная чистка:
  - удаление markdown-картинок `![...](...)`
  - удаление служебных OCR-разделителей страниц (если встретятся)
  - нормализация пробелов/переносов
  - удаление `\x00`

In [5]:
MD_IMAGE_RE = re.compile(r"!\[.*?\]\(.*?\)")
OCR_PAGE_SPLIT_RE = re.compile(r"\n\n\{(\d+)\}-{48}\n\n")
GLYPH_RE = re.compile(r"glyph<[^>]*>")
CAP_RE = re.compile(r"/([A-Z])\.cap")

SLASH_CMD_RE = re.compile(
    r"/(zero|one|two|three|four|five|six|seven|eight|nine|period|comma|colon|hyphen|percent|dollar|space|plus|minus|slash|asterisk|lparen|rparen|parenright|parenleft|wedge\.1_E)"
    r"(\.pl\.tnum|\.tnum\.pl|\.pl|\.tnum|\.case|\.sups)?"
)

_SLASH_MAP = {
    "zero": "0",
    "one": "1",
    "two": "2",
    "three": "3",
    "four": "4",
    "five": "5",
    "six": "6",
    "seven": "7",
    "eight": "8",
    "nine": "9",
    "period": ".",
    "comma": ",",
    "colon": ":",
    "hyphen": "-",
    "percent": "%",
    "dollar": "$",
    "space": " ",
    "plus": "+",
    "minus": "-",
    "slash": "/",
    "asterisk": "*",
    "lparen": "(",
    "rparen": ")",
    "parenleft": "(",
    "parenright": ")",
    "wedge.1_E": "^",
}

def repair_text(text: str) -> str:
    if not text:
        return ""
    text = SLASH_CMD_RE.sub(lambda m: _SLASH_MAP.get(m.group(1), ""), text)
    text = GLYPH_RE.sub("", text)
    text = CAP_RE.sub(lambda m: m.group(1), text)
    return text

def safe_cleanup_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\x00", "")
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    text = OCR_PAGE_SPLIT_RE.sub("\n\n", text)
    text = MD_IMAGE_RE.sub("", text)

    # trim trailing spaces per line
    text = "\n".join(line.rstrip() for line in text.split("\n"))

    # collapse excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # collapse multiple spaces (but keep newlines intact)
    text = re.sub(r"[ \t]{2,}", " ", text)

    return text.strip()

# Docling: pipeline config

Настраиваем Docling для PDF:
- OCR включён (auto)
- full-page OCR не форсируем
- извлечение структуры таблиц включено (accurate + cell matching)

In [6]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions, TableFormerMode,
    EasyOcrOptions,   # TODO: попробовать TesseractOcrOptions / TesseractCliOcrOptions
)

def make_docling_converter() -> DocumentConverter:
    opts = PdfPipelineOptions()

    opts.do_ocr = True
    
    if opts.ocr_options is None:
        opts.ocr_options = EasyOcrOptions(force_full_page_ocr=False)
    else:
        opts.ocr_options.force_full_page_ocr = False

    opts.do_table_structure = True
    opts.table_structure_options.do_cell_matching = True
    opts.table_structure_options.mode = TableFormerMode.ACCURATE

    return DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)}
    )

doc_converter = make_docling_converter()
print("Docling converter ready.")

Docling converter ready.


# Parsing: PDFs -> parsed_reports/{doc_id}.json

Для каждого doc_id:
- парсим `{doc_id}.pdf` через Docling
- сохраняем rich JSON в `parsed_reports/{doc_id}.json`
- если файл уже есть — пропускаем (resume-safe)
- ошибки логируем в `reports/parsing_errors.jsonl`

In [7]:
PARSED_DIR = DIRS["parsed_reports"]
ERRORS_PATH = DIRS["reports"] / "parsing_errors.jsonl"

def parse_pdf_to_rich_json(doc_id: str, pdf_path: Path, out_path: Path) -> dict:
    res = doc_converter.convert(str(pdf_path))
    doc = res.document
    if hasattr(doc, "export_to_dict"):
        return doc.export_to_dict()
    if hasattr(doc, "model_dump"):
        return doc.model_dump()
    if hasattr(doc, "dict"):
        return doc.dict()
    raise RuntimeError("Docling document does not support export_to_dict/model_dump/dict")

def append_jsonl(path: Path, row: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("ab") as f:
        f.write(orjson.dumps(row))
        f.write(b"\n")

def run_parsing_all(doc_id_list: List[str]) -> None:
    ok, skipped, failed = 0, 0, 0

    for doc_id in tqdm(doc_id_list, desc="Parsing PDFs"):
        pdf_path = Path(docs_meta[doc_id]["pdf_path"])
        out_path = PARSED_DIR / f"{doc_id}.json"

        if out_path.exists():
            skipped += 1
            continue

        try:
            rich = parse_pdf_to_rich_json(doc_id, pdf_path, out_path)
            write_json(out_path, rich)
            ok += 1
        except Exception as e:
            failed += 1
            append_jsonl(ERRORS_PATH, {
                "doc_id": doc_id,
                "pdf_path": str(pdf_path),
                "error": repr(e),
            })

    print(f"Parsed ok={ok}, skipped={skipped}, failed={failed}")
    if ERRORS_PATH.exists():
        print("Errors logged to:", ERRORS_PATH)

run_parsing_all(doc_id_list)

Parsing PDFs:   0%|          | 0/14 [00:00<?, ?it/s]

Parsed ok=14, skipped=0, failed=0


# Merging helpers: page mapping, filtering, formatting

Собираем page_text из структуры Docling:
- берём `texts[]` и `tables[]`, маппим по `prov[0].page_no`
- сортируем элементы по bbox (top, left)
- игнорируем footer-блоки (из `furniture[]`) и картинки (из `pictures[]`)
- нормализуем страницы `1..max_page_no` с вставкой пустых страниц
- формируем `page_text` с лёгкой структурой + применяем `repair_text` и `safe_cleanup_text`

In [8]:
def _get_first_prov(obj: dict) -> Optional[dict]:
    prov = obj.get("prov")
    if isinstance(prov, list) and prov:
        p0 = prov[0]
        if isinstance(p0, dict):
            return p0
    return None

def _get_page_no(obj: dict) -> Optional[int]:
    p0 = _get_first_prov(obj)
    if not p0:
        return None
    pn = p0.get("page_no")
    if isinstance(pn, int):
        return pn
    return None

def _get_bbox(obj: dict) -> Optional[dict]:
    p0 = _get_first_prov(obj)
    if not p0:
        return None
    bb = p0.get("bbox")
    return bb if isinstance(bb, dict) else None

def _bbox_sort_key(bb: Optional[dict]) -> Tuple[float, float]:
    if not bb:
        return (1e9, 1e9)
    t = bb.get("t", 1e9)
    l = bb.get("l", 1e9)
    try:
        return (float(t), float(l))
    except Exception:
        return (1e9, 1e9)

def _is_footer_like(furn: dict) -> bool:
    label = (furn.get("label") or "").lower()
    kind = (furn.get("kind") or "").lower()
    name = (furn.get("name") or "").lower()
    txt = (furn.get("text") or "").lower()
    s = " ".join([label, kind, name, txt])
    return ("footer" in s) or ("page_footer" in s) or ("footnote" in s)

def _collect_footer_bboxes(parsed: dict) -> Dict[int, List[dict]]:
    out: Dict[int, List[dict]] = {}
    furniture = parsed.get("furniture")
    if not isinstance(furniture, list):
        return out
    for furn in furniture:
        if not isinstance(furn, dict):
            continue
        if not _is_footer_like(furn):
            continue
        pn = _get_page_no(furn)
        if pn is None:
            continue
        bb = _get_bbox(furn)
        if not bb:
            continue
        out.setdefault(pn, []).append(bb)
    return out

def _bbox_inside_footer(bb: Optional[dict], footer_bbs: List[dict]) -> bool:
    if not bb or not footer_bbs:
        return False
    try:
        t = float(bb.get("t", 0.0))
        b = float(bb.get("b", 0.0))
        l = float(bb.get("l", 0.0))
        r = float(bb.get("r", 0.0))
    except Exception:
        return False

    for fbb in footer_bbs:
        try:
            ft = float(fbb.get("t", 0.0))
            fb = float(fbb.get("b", 0.0))
            fl = float(fbb.get("l", 0.0))
            fr = float(fbb.get("r", 0.0))
        except Exception:
            continue

        if (t >= ft and b <= fb and l >= fl and r <= fr):
            return True
    return False

def _table_to_html_or_text(tbl: dict) -> str:
    html = tbl.get("html")
    if isinstance(html, str) and html.strip():
        return html.strip()
    text = tbl.get("text")
    if isinstance(text, str) and text.strip():
        return text.strip()
    # minimal fallback (kept short)
    return json.dumps({k: tbl.get(k) for k in ("label", "id", "self_ref") if k in tbl}, ensure_ascii=False)

def _format_text_item(item: dict) -> str:
    txt = item.get("text")
    if not isinstance(txt, str):
        return ""
    return txt.strip()

def build_page_items(parsed: dict) -> Dict[int, List[dict]]:
    footer_bboxes = _collect_footer_bboxes(parsed)

    page_items: Dict[int, List[dict]] = {}

    texts = parsed.get("texts")
    if isinstance(texts, list):
        for it in texts:
            if not isinstance(it, dict):
                continue
            pn = _get_page_no(it)
            if pn is None:
                continue
            bb = _get_bbox(it)
            if _bbox_inside_footer(bb, footer_bboxes.get(pn, [])):
                continue
            s = _format_text_item(it)
            if not s:
                continue
            page_items.setdefault(pn, []).append({
                "kind": "text",
                "page_no": pn,
                "bbox": bb,
                "text": s,
            })

    tables = parsed.get("tables")
    if isinstance(tables, list):
        for tbl in tables:
            if not isinstance(tbl, dict):
                continue
            pn = _get_page_no(tbl)
            if pn is None:
                continue
            bb = _get_bbox(tbl)
            if _bbox_inside_footer(bb, footer_bboxes.get(pn, [])):
                continue
            html_or_text = _table_to_html_or_text(tbl)
            if not html_or_text:
                continue
            page_items.setdefault(pn, []).append({
                "kind": "table",
                "page_no": pn,
                "bbox": bb,
                "text": html_or_text,
            })

    for pn, items in page_items.items():
        items.sort(key=lambda x: _bbox_sort_key(x.get("bbox")))

    return page_items

def max_page_no_from_pages(parsed: dict) -> int:
    pages = parsed.get("pages")
    mx = 0

    if isinstance(pages, list):
        for p in pages:
            if not isinstance(p, dict):
                continue
            pn = p.get("page_no")
            if isinstance(pn, int) and pn > mx:
                mx = pn

    elif isinstance(pages, dict):
        # pages can be dict keyed by page index/string
        for _, p in pages.items():
            if not isinstance(p, dict):
                continue
            pn = p.get("page_no")
            if isinstance(pn, int) and pn > mx:
                mx = pn

    # fallback: derive from prov.page_no in texts/tables
    if mx == 0:
        for key in ("texts", "tables"):
            arr = parsed.get(key)
            if not isinstance(arr, list):
                continue
            for it in arr:
                if not isinstance(it, dict):
                    continue
                pn = _get_page_no(it)
                if isinstance(pn, int) and pn > mx:
                    mx = pn

    return mx

def normalize_pages_1_to_n(page_items: Dict[int, List[dict]], max_page_no: int) -> Dict[int, List[dict]]:
    out: Dict[int, List[dict]] = {}
    for pn in range(1, max_page_no + 1):
        out[pn] = page_items.get(pn, [])
    return out

def format_page_text(items: List[dict]) -> str:
    out: List[str] = []
    i = 0
    while i < len(items):
        it = items[i]
        kind = it["kind"]
        txt = it["text"]

        if kind == "text":
            s = txt.strip()
            # naive header heuristic (short line, no trailing period)
            if 0 < len(s) <= 80 and not s.endswith("."):
                out.append(f"## {s}")
            else:
                out.append(s)

            if s.endswith(":") and i + 1 < len(items) and items[i + 1]["kind"] == "table":
                out.append(items[i + 1]["text"].strip())
                i += 2
                continue

        elif kind == "table":
            out.append(txt.strip())

        i += 1

    return "\n\n".join([x for x in out if x and x.strip()])

def build_pages_text(parsed: dict) -> Tuple[List[dict], int, int]:
    page_items = build_page_items(parsed)
    max_pn = max_page_no_from_pages(parsed)
    page_items = normalize_pages_1_to_n(page_items, max_pn)

    pages_out: List[dict] = []
    empty_pages = 0

    for pn in range(1, max_pn + 1):
        items = page_items[pn]
        if not items:
            pages_out.append({"page_no": pn, "text": ""})
            empty_pages += 1
            continue

        page_text = format_page_text(items)
        page_text = repair_text(page_text)
        page_text = safe_cleanup_text(page_text)

        if not page_text.strip():
            pages_out.append({"page_no": pn, "text": ""})
            empty_pages += 1
        else:
            pages_out.append({"page_no": pn, "text": page_text})

    return pages_out, max_pn, empty_pages

# Merging: parsed_reports -> merged_reports

Для каждого doc_id:
- читаем `parsed_reports/{doc_id}.json`
- строим `merged_reports/{doc_id}.json`:
  - `metainfo`: subset-поля
  - `content.pages`: список `{page_no, text}` (1..N, с пустыми страницами)
- пустые страницы остаются пустыми

In [9]:
MERGED_DIR = DIRS["merged_reports"]

def build_merged_report(parsed: dict, doc_id: str) -> dict:
    pages, max_pn, empty_pages = build_pages_text(parsed)
    meta = dict(docs_meta.get(doc_id, {}))
    meta["doc_id"] = doc_id
    meta["num_pages"] = max_pn
    meta["num_empty_pages"] = empty_pages
    return {
        "metainfo": meta,
        "content": {
            "pages": pages
        }
    }

def run_merging_all(doc_id_list: List[str]) -> None:
    ok, skipped, failed = 0, 0, 0
    merge_errors = DIRS["reports"] / "merging_errors.jsonl"

    for doc_id in tqdm(doc_id_list, desc="Merging reports"):
        in_path = PARSED_DIR / f"{doc_id}.json"
        out_path = MERGED_DIR / f"{doc_id}.json"

        if out_path.exists():
            skipped += 1
            continue

        try:
            parsed = read_json(in_path)
            merged = build_merged_report(parsed, doc_id)
            write_json(out_path, merged)
            ok += 1
        except Exception as e:
            failed += 1
            append_jsonl(merge_errors, {
                "doc_id": doc_id,
                "error": repr(e),
            })

    print(f"Merged ok={ok}, skipped={skipped}, failed={failed}")
    if merge_errors.exists():
        print("Errors logged to:", merge_errors)

run_merging_all(doc_id_list)

Merging reports:   0%|          | 0/14 [00:00<?, ?it/s]

Merged ok=14, skipped=0, failed=0


In [10]:
SECTIONED_DIR = DIRS["sectioned_reports"]

def normalize_ws(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "")).strip()

def get_trimmed_page_bounds(merged: dict) -> Tuple[Optional[int], Optional[int]]:
    pages = merged.get("content", {}).get("pages", [])
    page_nos = sorted(
        int(p["page_no"])
        for p in pages
        if isinstance(p, dict) and isinstance(p.get("page_no"), int)
    )
    if not page_nos:
        return None, None

    min_page = page_nos[0]
    max_page = page_nos[-1]

    allowed_start = min_page + TRIM_FRONT_PAGES
    allowed_end = max_page - TRIM_BACK_PAGES

    if allowed_start > allowed_end:
        return None, None

    return allowed_start, allowed_end

def get_page_map_from_merged(merged: dict) -> Dict[int, str]:
    pages = merged.get("content", {}).get("pages", [])

    raw_pages = {
        int(p["page_no"]): p.get("text", "")
        for p in pages
        if isinstance(p, dict) and isinstance(p.get("page_no"), int)
    }
    if not raw_pages:
        return {}

    allowed_start, allowed_end = get_trimmed_page_bounds(merged)
    if allowed_start is None or allowed_end is None:
        return {}

    return {
        pn: txt
        for pn, txt in raw_pages.items()
        if allowed_start <= pn <= allowed_end
    }

def looks_like_toc_line(text: str) -> bool:
    text = normalize_ws(text)
    if not text:
        return False

    low = text.lower()

    if re.search(r"\.{3,}\s*\d{1,4}$", text):
        return True
    if re.search(r"\s{2,}\d{1,4}$", text):
        return True
    if low in {"содержание", "оглавление", "contents", "table of contents"}:
        return True

    return False

def looks_like_figure_or_table_caption(text: str) -> bool:
    text = normalize_ws(text)
    low = text.lower()

    if low.startswith(("figure ", "table ", "рис.", "табл.", "scheme ", "схема ")):
        return True

    if re.match(r"^(рис|рисунок|таблица|table|figure)\s*[\.\-]?\s*\d+", low):
        return True

    return False

def looks_like_heading(text: str) -> bool:
    text = normalize_ws(text)
    if not text:
        return False

    low = text.lower()

    # базовый отсев мусора
    if len(text) < 2 or len(text) > 220:
        return False

    if looks_like_toc_line(text):
        return False

    if looks_like_figure_or_table_caption(text):
        return False

    # длинные законченные предложения обычно не заголовки
    if text.endswith(".") and len(text) > 80:
        return False

    # приложения / appendix — безопасный универсальный паттерн
    if re.match(r"^приложение(\s+[а-яa-z0-9\-]+)?$", low):
        return True
    if re.match(r"^appendix(\s+[a-z0-9\-]+)?$", low):
        return True

    # нумерованные заголовки: 1, 1.1, 1.1.1, 2.3-
    if re.match(r"^\d+(\.\d+){0,5}([.)\-]|\s)\s*\S+", text):
        return True

    # римские заголовки
    if re.match(r"^[IVXLCM]+\.\s+\S+", text):
        return True

    # буквенные заголовки вида А) ... / A) ...
    if re.match(r"^[А-ЯA-Z]\)\s+\S+", text):
        return True

    # короткая строка без точки в конце часто является заголовком
    if len(text) <= 90 and not text.endswith("."):
        return True

    # ВЕРХНИЙ РЕГИСТР часто встречается в заголовках
    letters = [ch for ch in text if ch.isalpha()]
    if letters:
        upper_ratio = sum(1 for ch in letters if ch.isupper()) / len(letters)
        if upper_ratio >= 0.7 and len(text) <= 120:
            return True

    return False

def heading_score(text: str, label: str) -> float:
    text = normalize_ws(text)
    low = text.lower()
    score = 0.0

    # сильный сигнал от парсера
    if "heading" in label or "title" in label or "section_header" in label:
        score += 5.0

    # структурный сигнал: нумерация раздела
    if re.match(r"^\d+(\.\d+){0,5}([.)\-]|\s)\s*\S+", text):
        score += 4.0

    # универсальный сигнал приложений
    if re.match(r"^приложение(\s+[а-яa-z0-9\-]+)?$", low):
        score += 2.5
    if re.match(r"^appendix(\s+[a-z0-9\-]+)?$", low):
        score += 2.5

    # короткие строки чаще бывают заголовками
    if len(text) <= 80:
        score += 1.5
    elif len(text) <= 140:
        score += 0.5

    # отсутствие точки в конце тоже слабый плюс
    if not text.endswith("."):
        score += 1.0

    # штрафы за сомнительные строки
    if looks_like_toc_line(text):
        score -= 5.0
    if looks_like_figure_or_table_caption(text):
        score -= 5.0

    return score

def detect_heading_candidates(parsed: dict) -> List[dict]:
    candidates = []
    texts = parsed.get("texts", [])
    if not isinstance(texts, list):
        return candidates

    for it in texts:
        if not isinstance(it, dict):
            continue

        label = str(it.get("label", "")).lower()
        text = normalize_ws(it.get("text", ""))
        if not text:
            continue

        label_is_heading = (
            "heading" in label or "title" in label or "section_header" in label
        )

        if not looks_like_heading(text) and not label_is_heading:
            continue

        pn = _get_page_no(it)
        if pn is None:
            continue

        score = heading_score(text, label)
        if score <= 0:
            continue

        candidates.append({
            "page_no": int(pn),
            "title": text[:300],
            "score": score,
        })

    return candidates

def build_fallback_sections_from_pages(merged: dict, doc_id: str) -> List[dict]:
    page_map = get_page_map_from_merged(merged)
    page_nos = sorted(page_map.keys())
    if not page_nos:
        return []

    sections = []
    step = max(1, MAX_SECTION_PAGES_FALLBACK - FALLBACK_PAGE_OVERLAP)

    start_idx = 0
    sec_num = 0
    while start_idx < len(page_nos):
        chunk_pages = page_nos[start_idx:start_idx + MAX_SECTION_PAGES_FALLBACK]
        if not chunk_pages:
            break

        start_page = chunk_pages[0]
        end_page = chunk_pages[-1]

        texts = []
        for pn in chunk_pages:
            txt = normalize_ws(page_map.get(pn, ""))
            if txt:
                texts.append(txt)

        section_text = "\n\n".join(texts).strip()
        if section_text:
            sections.append({
                "section_id": f"{doc_id}_sec_{sec_num:04d}",
                "doc_id": doc_id,
                "title": f"Pages {start_page}-{end_page}",
                "start_page": start_page,
                "end_page": end_page,
                "text": section_text,
                "is_fallback_window": True,
            })
            sec_num += 1

        start_idx += step

    return sections

def build_sections_from_headings(parsed: dict, merged: dict, doc_id: str) -> List[dict]:
    page_map = get_page_map_from_merged(merged)
    all_pages = sorted(page_map.keys())
    if not all_pages:
        return []

    allowed_pages = set(all_pages)

    headings = detect_heading_candidates(parsed)
    headings = [h for h in headings if int(h["page_no"]) in allowed_pages]
    if not headings:
        return []

    # Оставляем один лучший heading на страницу, так как sectioning page-level
    best_by_page: Dict[int, dict] = {}
    for h in headings:
        pn = int(h["page_no"])
        prev = best_by_page.get(pn)
        if prev is None or h["score"] > prev["score"] or (
            h["score"] == prev["score"] and len(h["title"]) < len(prev["title"])
        ):
            best_by_page[pn] = h

    headings = sorted(best_by_page.values(), key=lambda x: (x["page_no"], x["title"]))
    if not headings:
        return []

    sections = []
    sec_num = 0

    # intro block до первого heading
    first_heading_page = int(headings[0]["page_no"])
    if all_pages[0] < first_heading_page:
        intro_start = all_pages[0]
        intro_end = first_heading_page - 1

        texts = []
        for pn in range(intro_start, intro_end + 1):
            txt = normalize_ws(page_map.get(pn, ""))
            if txt:
                texts.append(txt)

        intro_text = "\n\n".join(texts).strip()
        if intro_text:
            sections.append({
                "section_id": f"{doc_id}_sec_{sec_num:04d}",
                "doc_id": doc_id,
                "title": f"Pages {intro_start}-{intro_end}",
                "start_page": intro_start,
                "end_page": intro_end,
                "text": intro_text,
                "is_fallback_window": False,
            })
            sec_num += 1

    for i, h in enumerate(headings):
        start_page = int(h["page_no"])

        if i + 1 < len(headings):
            next_page = int(headings[i + 1]["page_no"])
            end_page = next_page - 1
        else:
            end_page = all_pages[-1]

        if end_page < start_page:
            end_page = start_page

        texts = []
        for pn in range(start_page, end_page + 1):
            txt = normalize_ws(page_map.get(pn, ""))
            if txt:
                texts.append(txt)

        section_text = "\n\n".join(texts).strip()
        if not section_text:
            continue

        sections.append({
            "section_id": f"{doc_id}_sec_{sec_num:04d}",
            "doc_id": doc_id,
            "title": str(h["title"])[:300],
            "start_page": start_page,
            "end_page": end_page,
            "text": section_text,
            "is_fallback_window": False,
            "heading_score": float(h.get("score", 0.0)),
        })
        sec_num += 1

    return sections

def sections_page_coverage(sections: List[dict]) -> set:
    covered = set()
    for s in sections:
        sp = s.get("start_page")
        ep = s.get("end_page")
        if isinstance(sp, int) and isinstance(ep, int):
            for pn in range(sp, ep + 1):
                covered.add(pn)
    return covered

def _section_char_len(sec: dict) -> int:
    return len((sec.get("text", "") or "").strip())

def _section_page_span(sec: dict) -> int:
    sp = sec.get("start_page")
    ep = sec.get("end_page")
    if isinstance(sp, int) and isinstance(ep, int) and ep >= sp:
        return ep - sp + 1
    return 0

def _merge_section_group(group: List[dict], doc_id: str, merged_idx: int) -> dict:
    assert group, "group must be non-empty"

    start_page = group[0]["start_page"]
    end_page = group[-1]["end_page"]

    texts = []
    titles = []
    is_fallback = False

    for sec in group:
        txt = (sec.get("text", "") or "").strip()
        if txt:
            texts.append(txt)
        t = (sec.get("title", "") or "").strip()
        if t:
            titles.append(t)
        is_fallback = is_fallback or bool(sec.get("is_fallback_window", False))

    merged_text = "\n\n".join(texts).strip()
    merged_title = titles[0] if titles else f"Pages {start_page}-{end_page}"

    return {
        "section_id": f"{doc_id}_sec_{merged_idx:04d}",
        "doc_id": doc_id,
        "title": merged_title,
        "start_page": start_page,
        "end_page": end_page,
        "text": merged_text,
        "is_fallback_window": is_fallback,
        "source_section_count": len(group),
    }

def merge_small_neighbor_sections(sections: List[dict], doc_id: str) -> List[dict]:
    if not sections:
        return []

    merged = []
    buffer_group = []

    def flush_buffer():
        nonlocal buffer_group, merged
        if buffer_group:
            merged.append(_merge_section_group(buffer_group, doc_id, len(merged)))
            buffer_group = []

    for sec in sections:
        if not buffer_group:
            buffer_group = [sec]
            continue

        current_chars = sum(_section_char_len(s) for s in buffer_group)
        current_pages = sum(_section_page_span(s) for s in buffer_group)

        should_force_flush = (
            current_chars >= MAX_MERGED_SECTION_CHARS
            or current_pages >= MAX_MERGED_SECTION_PAGES
        )

        if should_force_flush:
            flush_buffer()
            buffer_group = [sec]
            continue

        if current_chars < MIN_MERGED_SECTION_CHARS or current_pages < MIN_MERGED_SECTION_PAGES:
            buffer_group.append(sec)
            continue

        flush_buffer()
        buffer_group = [sec]

    flush_buffer()

    if len(merged) >= 2:
        last = merged[-1]
        if (
            _section_char_len(last) < MIN_MERGED_SECTION_CHARS
            and _section_page_span(last) < MIN_MERGED_SECTION_PAGES
        ):
            prev = merged[-2]
            regroup = [prev, last]
            merged = merged[:-2]
            merged.append(_merge_section_group(regroup, doc_id, len(merged)))

    return merged

def extract_sections(parsed: dict, merged: dict, doc_id: str) -> List[dict]:
    page_map = get_page_map_from_merged(merged)
    all_pages = set(page_map.keys())
    if not all_pages:
        return []

    sections = build_sections_from_headings(parsed, merged, doc_id)

    if sections:
        covered = sections_page_coverage(sections)
        coverage_ratio = len(covered & all_pages) / max(1, len(all_pages))
        if coverage_ratio < 0.55:
            sections = build_fallback_sections_from_pages(merged, doc_id)
    else:
        sections = build_fallback_sections_from_pages(merged, doc_id)

    sections = merge_small_neighbor_sections(sections, doc_id)
    return sections

чек на число разбивок по заголовкам

In [11]:
section_stats = []

for doc_id in doc_id_list:
    parsed = read_json(PARSED_DIR / f"{doc_id}.json")
    merged = read_json(MERGED_DIR / f"{doc_id}.json")

    all_raw_pages = merged.get("content", {}).get("pages", [])
    raw_page_nos = sorted(
        int(p["page_no"])
        for p in all_raw_pages
        if isinstance(p, dict) and isinstance(p.get("page_no"), int)
    )

    trimmed_page_map = get_page_map_from_merged(merged)
    trimmed_page_nos = sorted(trimmed_page_map.keys())

    raw_sections = build_sections_from_headings(parsed, merged, doc_id)
    fallback_sections = build_fallback_sections_from_pages(merged, doc_id)
    final_sections = extract_sections(parsed, merged, doc_id)

    section_stats.append({
        "doc_id": doc_id,
        "raw_pages": len(raw_page_nos),
        "trimmed_pages": len(trimmed_page_nos),
        "heading_sections_before_merge": len(raw_sections),
        "fallback_sections_before_merge": len(fallback_sections),
        "final_sections_after_merge": len(final_sections),
        "first_trimmed_page": trimmed_page_nos[0] if trimmed_page_nos else None,
        "last_trimmed_page": trimmed_page_nos[-1] if trimmed_page_nos else None,
    })

df_sections = pd.DataFrame(section_stats).sort_values("final_sections_after_merge", ascending=False)
df_sections

,doc_id,raw_pages,trimmed_pages,heading_sections_before_merge,fallback_sections_before_merge,final_sections_after_merge,first_trimmed_page,last_trimmed_page
13,sto_gazprom_1.1-2009,84,84,84,17,38,1,84
5,Gost_7-32_2017,47,47,47,10,19,1,47
10,sto-gazprom-1.15-2014,40,40,40,8,19,1,40
12,sto_gazprom_1.0-2009,41,41,41,9,19,1,41
4,GOST_R_2.105-2019,36,36,35,8,15,1,36
3,4293767636,28,28,28,6,13,1,28
6,cto_gazprom_1.13-2012,32,32,32,7,13,1,32
9,rosstandart_290725-gost,36,36,34,7,12,1,36
11,sto-gazprom-1.6-2014,24,24,24,5,10,1,24
2,34.602-2020,12,12,12,3,6,1,12


In [12]:
print("Total final sections after merge:", int(df_sections["final_sections_after_merge"].sum()))
print("Total trimmed pages:", int(df_sections["trimmed_pages"].sum()))

Total final sections after merge: 176
Total trimmed pages: 406


In [13]:
# err_path = DIRS["reports"] / "sectioning_errors.jsonl"
# if err_path.exists():
#     err_path.unlink()
#     print("Deleted old error log:", err_path)

# for p in SECTIONED_DIR.glob("*.json"):
#     p.unlink()
# print("Cleared sectioned_reports")

In [14]:
# # Clear downstream artifacts before re-running section summaries

# # 1) sectioned reports
# for p in DIRS["sectioned_reports"].glob("*.json"):
#     p.unlink()

# # 2) chunked reports
# for p in DIRS["chunked_reports"].glob("*.json"):
#     p.unlink()

# # 3) vector DBs
# for p in DIRS["vector_dbs"].glob("*"):
#     if p.is_file():
#         p.unlink()

# # 4) embedding caches built from sections/chunks
# for p in DIRS["section_emb_cache"].glob("*"):
#     if p.is_file():
#         p.unlink()

# for p in DIRS["chunk_emb_cache"].glob("*"):
#     if p.is_file():
#         p.unlink()

# # 5) logs
# for log_name in [
#     "sectioning_errors.jsonl",
#     "chunking_errors.jsonl",
# ]:
#     log_path = DIRS["reports"] / log_name
#     if log_path.exists():
#         log_path.unlink()

# print("Cleared sectioned_reports, chunked_reports, vector_dbs, section/chunk embedding caches, and logs")

In [15]:
import asyncio
from typing import List
from openai import AsyncOpenAI
from tqdm.auto import tqdm

# отдельный async-клиент, не перетираем sync client
openrouter_async_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MAX_CONCURRENT_SECTION_REQUESTS = SECTION_SUMMARY_CONCURRENCY
_section_summary_semaphore = asyncio.Semaphore(MAX_CONCURRENT_SECTION_REQUESTS)


def truncate_for_summary(text: str, max_chars: int = MAX_SECTION_SUMMARY_INPUT_CHARS) -> str:
    text = (text or "").strip()
    if len(text) <= max_chars:
        return text

    half = max_chars // 2
    head = text[:half].strip()
    tail = text[-half:].strip()
    return f"{head}\n\n[...]\n\n{tail}"


def make_section_summary_prompt(title: str, text: str) -> str:
    return f"""
Ты создаешь краткое описание раздела для retrieval-oriented базы знаний по технической документации Газпрома.

Контекст корпуса:
- стандарты (ГОСТ, СТО, РД, ТУ и др.)
- регламенты и нормативные документы
- производственные и технологические инструкции
- требования безопасности и охраны труда
- порядок выполнения работ
- технические требования, параметры, нормы, ограничения
- документация по оборудованию, процессам и процедурам

Задача:
Напиши компактное, плотное по смыслу описание раздела так, чтобы по нему хорошо работал семантический поиск.

ОСОБО ВАЖНО:
- Обязательно сохраняй и явно упоминай номера стандартов и документов (например: ГОСТ 123-45, СТО Газпром 1.2-2020 и т.д.), если они есть в тексте.
- Не теряй технические обозначения, индексы, коды, сокращения.
- Если в разделе есть ссылки на нормативные документы — отрази это.

Что нужно отразить:
- о каком объекте, процессе, оборудовании или документе идет речь
- какие требования, нормы, ограничения или условия задаются
- какие действия, процедуры или этапы описаны
- какие термины, параметры, классификации или критерии используются
- какие меры безопасности или обязательные условия указаны

Требования к ответу:
- Пиши только на русском языке
- Не выдумывай ничего сверх текста
- Не пиши "в данном разделе"
- Не делай списки и markdown
- 2–5 плотных предложений
- Сохраняй формулировки максимально близко к техническому стилю

Название раздела:
{title}

Текст раздела:
{text}
""".strip()


async def summarize_section(title: str, text: str, model: str = SECTION_SUMMARY_MODEL, max_retries: int = 3) -> str:
    text = truncate_for_summary(text)
    prompt = make_section_summary_prompt(title, text)

    last_err = None

    for attempt in range(max_retries):
        try:
            async with _section_summary_semaphore:
                response = await openrouter_async_client.chat.completions.create(
                    model=model,
                    messages=[
                        {
                            "role": "system",
                            "content": (
                                "Ты делаешь краткие retrieval-oriented summaries для корпуса "
                                "технической и нормативной документации на русском языке."
                            ),
                        },
                        {"role": "user", "content": prompt},
                    ],
                    temperature=0.2,
                )

            msg = response.choices[0].message
            content = (msg.content or "").strip()

            if not content:
                raise ValueError("Empty summary response from model")

            return normalize_ws(content)

        except Exception as e:
            last_err = e
            if attempt < max_retries - 1:
                await asyncio.sleep(2 * (attempt + 1))
            else:
                raise last_err


async def _summarize_one_section(index: int, sec: dict, model: str = SECTION_SUMMARY_MODEL):
    sec_text = sec.get("text", "") or ""
    sec_title = sec.get("title", "") or "Untitled section"
    summary = await summarize_section(sec_title, sec_text, model=model)
    return index, summary


async def build_sectioned_report(parsed: dict, merged: dict, doc_id: str) -> dict:
    sections = extract_sections(parsed, merged, doc_id)

    if not sections:
        raise ValueError(f"No sections extracted for doc_id={doc_id}")

    tasks = [
        asyncio.create_task(_summarize_one_section(idx, sec))
        for idx, sec in enumerate(sections)
    ]

    summaries_by_index = [None] * len(sections)

    for fut in tqdm(
        asyncio.as_completed(tasks),
        total=len(tasks),
        desc=f"Summarizing sections: {doc_id}",
        leave=False,
    ):
        idx, summary = await fut
        summaries_by_index[idx] = summary

    for sec, summary in zip(sections, summaries_by_index):
        sec["summary"] = summary

    return {
        "metainfo": merged.get("metainfo", {}),
        "content": {
            "pages": merged.get("content", {}).get("pages", []),
            "sections": sections,
        }
    }


async def run_sectioning_all(doc_id_list: List[str]) -> None:
    ok, skipped, failed = 0, 0, 0
    section_errors = DIRS["reports"] / "sectioning_errors.jsonl"

    for doc_id in tqdm(doc_id_list, desc="Sectioning + summarization"):
        parsed_path = PARSED_DIR / f"{doc_id}.json"
        merged_path = MERGED_DIR / f"{doc_id}.json"
        out_path = SECTIONED_DIR / f"{doc_id}.json"

        if out_path.exists():
            skipped += 1
            continue

        try:
            parsed = read_json(parsed_path)
            merged = read_json(merged_path)
            sectioned = await build_sectioned_report(parsed, merged, doc_id)
            write_json(out_path, sectioned)
            ok += 1
        except Exception as e:
            failed += 1
            append_jsonl(section_errors, {
                "doc_id": doc_id,
                "error": repr(e),
            })

    print(f"Sectioned ok={ok}, skipped={skipped}, failed={failed}")
    if section_errors.exists():
        print("Errors logged to:", section_errors)


await run_sectioning_all(doc_id_list)

Sectioning + summarization:   0%|          | 0/14 [00:00<?, ?it/s]

Summarizing sections: 2019-09-17-safety-policy:   0%|          | 0/1 [00:00<?, ?it/s]

Summarizing sections: 34.201-2020:   0%|          | 0/5 [00:00<?, ?it/s]

Summarizing sections: 34.602-2020:   0%|          | 0/6 [00:00<?, ?it/s]

Summarizing sections: 4293767636:   0%|          | 0/13 [00:00<?, ?it/s]

...

Summarizing sections: sto-gazprom-1.15-2014:   0%|          | 0/19 [00:00<?, ?it/s]

Summarizing sections: sto-gazprom-1.6-2014:   0%|          | 0/10 [00:00<?, ?it/s]

Summarizing sections: sto_gazprom_1.0-2009:   0%|          | 0/19 [00:00<?, ?it/s]

Summarizing sections: sto_gazprom_1.1-2009:   0%|          | 0/38 [00:00<?, ?it/s]

Sectioned ok=14, skipped=0, failed=0


# Chunking config

Создаём сплиттер (RecursiveCharacterTextSplitter) и функцию подсчёта токенов. Чанкаем постранично, пустые страницы пропускаем.

In [16]:
SPLITTER = make_splitter()

def token_len(text: str) -> int:
    return len(ENC.encode(text))

# Chunking: merged_reports → chunked_reports

Для каждого doc_id:
- читаем merged report
- для каждой страницы:
  - если текст пустой → пропускаем
  - иначе сплиттером режем на чанки
- каждому чанку назначаем:
  - chunk_id_num (0..N-1 по документу)
  - chunk_id (stable string)
  - page_no (1-based)
  - length_tokens (после repair+cleanup; считаем на тексте чанка)
  - type="content"
- сохраняем chunked_report

In [17]:
CHUNKED_DIR = DIRS["chunked_reports"]

def build_chunked_report(sectioned: dict, doc_id: str) -> dict:
    pages = sectioned.get("content", {}).get("pages", [])
    sections = sectioned.get("content", {}).get("sections", [])
    assert isinstance(sections, list)

    chunks: List[dict] = []
    chunk_id_num = 0

    for sec_idx, sec in enumerate(sections):
        if not isinstance(sec, dict):
            continue

        section_id = sec.get("section_id")
        title = sec.get("title", "")
        text = sec.get("text", "")
        start_page = sec.get("start_page")
        end_page = sec.get("end_page")

        if not isinstance(section_id, str) or not section_id:
            continue
        if not isinstance(text, str) or not text.strip():
            continue

        parts = SPLITTER.split_text(text)
        for idx, part in enumerate(parts):
            part = part.strip()
            if not part:
                continue
            lt = token_len(part)
            if lt == 0:
                continue

            chunks.append({
                "chunk_id_num": chunk_id_num,
                "chunk_id": f"{section_id}_chunk_{idx:03d}",
                "doc_id": doc_id,
                "section_id": section_id,
                "section_title": title,
                "page_start": start_page,
                "page_end": end_page,
                "text": part,
                "length_tokens": lt,
                "type": "content",
            })
            chunk_id_num += 1

    out = {
        "metainfo": sectioned.get("metainfo", {}),
        "content": {
            "pages": pages,
            "sections": sections,
            "chunks": chunks,
        }
    }
    return out

def run_chunking_all(doc_id_list: List[str]) -> None:
    ok, skipped, failed = 0, 0, 0
    chunk_errors = DIRS["reports"] / "chunking_errors.jsonl"

    for doc_id in tqdm(doc_id_list, desc="Chunking reports"):
        in_path = SECTIONED_DIR / f"{doc_id}.json"
        out_path = CHUNKED_DIR / f"{doc_id}.json"

        if out_path.exists():
            skipped += 1
            continue

        try:
            sectioned = read_json(in_path)
            chunked = build_chunked_report(sectioned, doc_id)
            write_json(out_path, chunked)
            ok += 1
        except Exception as e:
            failed += 1
            append_jsonl(chunk_errors, {"doc_id": doc_id, "error": repr(e)})

    print(f"Chunked ok={ok}, skipped={skipped}, failed={failed}")
    if chunk_errors.exists():
        print("Errors logged to:", chunk_errors)

run_chunking_all(doc_id_list)

Chunking reports:   0%|          | 0/14 [00:00<?, ?it/s]

Chunked ok=14, skipped=0, failed=0


# Embeddings

Считаем эмбеддинги для чанков через `sentence-transformers` (BAAI/bge-m3).
Вектора L2-нормализуем.

In [18]:
EMB_CACHE_DIR = DIRS["emb_cache"]
SECTION_EMB_CACHE_DIR = DIRS["section_emb_cache"]
CHUNK_EMB_CACHE_DIR = DIRS["chunk_emb_cache"]

def l2_normalize(mat: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms = np.maximum(norms, eps)
    return mat / norms

def embed_texts_local(texts: List[str], batch_size: int = EMBED_BATCH_SIZE) -> np.ndarray:
    emb = embedder.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=False,
    )
    if emb.dtype != np.float32:
        emb = emb.astype(np.float32, copy=False)
    emb = l2_normalize(emb)
    return emb

def build_section_embedding_text(sec: dict) -> str:
    title = normalize_ws(sec.get("title", ""))
    summary = normalize_ws(sec.get("summary", ""))
    return f"[TITLE]\n{title}\n\n[SUMMARY]\n{summary}".strip()

def collect_all_sections(doc_id_list: List[str]) -> List[dict]:
    rows = []
    for doc_id in doc_id_list:
        sectioned = read_json(SECTIONED_DIR / f"{doc_id}.json")
        sections = sectioned.get("content", {}).get("sections", [])
        for sec in sections:
            if not isinstance(sec, dict):
                continue
            rows.append(sec)
    return rows

def collect_all_chunks(doc_id_list: List[str]) -> List[dict]:
    rows = []
    for doc_id in doc_id_list:
        chunked = read_json(CHUNKED_DIR / f"{doc_id}.json")
        chunks = chunked.get("content", {}).get("chunks", [])
        for ch in chunks:
            if not isinstance(ch, dict):
                continue
            rows.append(ch)
    return rows

def load_or_build_global_section_embeddings(sections: List[dict]) -> np.ndarray:
    cache_path = SECTION_EMB_CACHE_DIR / "sections.npy"
    if cache_path.exists():
        arr = np.load(cache_path)
        if arr.dtype != np.float32:
            arr = arr.astype(np.float32, copy=False)
        return arr

    texts = [build_section_embedding_text(sec) for sec in sections]
    if not texts:
        arr = np.zeros((0, EMBED_DIM), dtype=np.float32)
    else:
        arr = embed_texts_local(texts, batch_size=EMBED_BATCH_SIZE)

    np.save(cache_path, arr)
    return arr

def load_or_build_global_chunk_embeddings(chunks: List[dict]) -> np.ndarray:
    cache_path = CHUNK_EMB_CACHE_DIR / "chunks.npy"
    if cache_path.exists():
        arr = np.load(cache_path)
        if arr.dtype != np.float32:
            arr = arr.astype(np.float32, copy=False)
        return arr

    texts = [str(ch.get("text", "")) for ch in chunks]
    if not texts:
        arr = np.zeros((0, EMBED_DIM), dtype=np.float32)
    else:
        arr = embed_texts_local(texts, batch_size=EMBED_BATCH_SIZE)

    np.save(cache_path, arr)
    return arr

# FAISS: build vector DB per document (IndexFlatIP)

Для каждого doc_id:
- читаем chunked_report
- получаем embeddings (из кэша или считаем)
- строим `IndexFlatIP`
- сохраняем:
  - `{doc_id}.faiss`
  - `{doc_id}.meta.json` (порядок = chunk_id_num)

In [19]:
VECTOR_DIR = DIRS["vector_dbs"]

def build_faiss_index_ip(vectors: np.ndarray) -> faiss.Index:
    if vectors.ndim != 2:
        raise ValueError("vectors must be 2D")
    dim = vectors.shape[1]
    index = faiss.IndexFlatIP(dim)
    if vectors.shape[0] > 0:
        index.add(vectors)
    return index

def save_faiss_index(path: Path, index: faiss.Index) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    faiss.write_index(index, str(path))

def build_and_save_global_vector_dbs(doc_id_list: List[str]) -> None:
    sections = collect_all_sections(doc_id_list)
    chunks = collect_all_chunks(doc_id_list)

    section_vectors = load_or_build_global_section_embeddings(sections)
    chunk_vectors = load_or_build_global_chunk_embeddings(chunks)

    sections_faiss = VECTOR_DIR / "sections.faiss"
    sections_meta = VECTOR_DIR / "sections.meta.json"

    chunks_faiss = VECTOR_DIR / "chunks.faiss"
    chunks_meta = VECTOR_DIR / "chunks.meta.json"

    sec_index = build_faiss_index_ip(section_vectors)
    ch_index = build_faiss_index_ip(chunk_vectors)

    save_faiss_index(sections_faiss, sec_index)
    save_faiss_index(chunks_faiss, ch_index)

    sec_meta = []
    for idx, sec in enumerate(sections):
        sec_meta.append({
            "index_pos": idx,
            "doc_id": sec["doc_id"],
            "section_id": sec["section_id"],
            "title": sec.get("title", ""),
            "start_page": sec.get("start_page"),
            "end_page": sec.get("end_page"),
            "summary": sec.get("summary", ""),
            "is_fallback_window": bool(sec.get("is_fallback_window", False)),
        })

    ch_meta = []
    for idx, ch in enumerate(chunks):
        ch_meta.append({
            "index_pos": idx,
            "chunk_id_num": int(ch["chunk_id_num"]),
            "chunk_id": ch["chunk_id"],
            "doc_id": ch["doc_id"],
            "section_id": ch["section_id"],
            "section_title": ch.get("section_title", ""),
            "page_start": ch.get("page_start"),
            "page_end": ch.get("page_end"),
            "length_tokens": int(ch.get("length_tokens", 0)),
            "type": ch.get("type", "content"),
        })

    write_json(sections_meta, sec_meta)
    write_json(chunks_meta, ch_meta)

build_and_save_global_vector_dbs(doc_id_list)

print("Saved:")
print("-", VECTOR_DIR / "sections.faiss")
print("-", VECTOR_DIR / "sections.meta.json")
print("-", VECTOR_DIR / "chunks.faiss")
print("-", VECTOR_DIR / "chunks.meta.json")

Saved:
- /kaggle/working/artifacts/vector_dbs/sections.faiss
- /kaggle/working/artifacts/vector_dbs/sections.meta.json
- /kaggle/working/artifacts/vector_dbs/chunks.faiss
- /kaggle/working/artifacts/vector_dbs/chunks.meta.json


# Reports + manifest

Собираем:
- `ingestion_report.csv`
- `manifest.json` (включая embedding_dim)

In [20]:
REPORTS_DIR = DIRS["reports"]

def compute_doc_stats(sectioned: dict, chunked: dict) -> dict:
    pages = sectioned.get("content", {}).get("pages", [])
    sections = sectioned.get("content", {}).get("sections", [])
    chunks = chunked.get("content", {}).get("chunks", [])

    lengths = [c["length_tokens"] for c in chunks if isinstance(c.get("length_tokens"), int)]
    return {
        "num_pages": len(pages),
        "num_sections": len(sections),
        "num_chunks": len(chunks),
        "num_empty_pages": sum(1 for p in pages if not p.get("text")),
        "avg_chunk_tokens": float(np.mean(lengths)) if lengths else 0.0,
        "max_chunk_tokens": int(max(lengths)) if lengths else 0,
    }

def build_ingestion_report(doc_id_list: List[str]) -> pd.DataFrame:
    rows = []
    for doc_id in doc_id_list:
        sectioned = read_json(SECTIONED_DIR / f"{doc_id}.json")
        chunked = read_json(CHUNKED_DIR / f"{doc_id}.json")
        stats = compute_doc_stats(sectioned, chunked)
        row = {
            "doc_id": doc_id,
            **stats,
        }
        rows.append(row)
    return pd.DataFrame(rows)

def build_manifest(doc_id_list: List[str], df_stats: pd.DataFrame) -> dict:
    docs = {}
    for doc_id in doc_id_list:
        meta = dict(docs_meta.get(doc_id, {}))
        stats_row = df_stats[df_stats.doc_id == doc_id].iloc[0].to_dict()
        docs[doc_id] = {
            "meta": meta,
            "stats": stats_row,
        }

    return {
        "pipeline_version": PIPELINE_VERSION,
        "chunking": {
            "chunk_size": CHUNK_SIZE,
            "chunk_overlap": CHUNK_OVERLAP,
            "tokenizer_model": SPLITTER_MODEL_NAME,
        },
        "sections": {
            "summary_model": SECTION_SUMMARY_MODEL,
            "fallback_max_pages": MAX_SECTION_PAGES_FALLBACK,
            "fallback_overlap_pages": FALLBACK_PAGE_OVERLAP,
        },
        "embeddings": {
            "model": EMBEDDING_MODEL,
            "dim": EMBED_DIM,
            "normalized": True,
        },
        "indices": {
            "sections_index": "vector_dbs/sections.faiss",
            "chunks_index": "vector_dbs/chunks.faiss",
            "metric": "inner_product",
        },
        "page_base": PAGE_BASE,
        "documents": docs,
    }

df_report = build_ingestion_report(doc_id_list)
report_csv_path = REPORTS_DIR / "ingestion_report.csv"
df_report.to_csv(report_csv_path, index=False)

manifest = build_manifest(doc_id_list, df_report)
write_json(ARTIFACTS_ROOT / "manifest.json", manifest)

print("Saved:")
print("-", report_csv_path)
print("-", ARTIFACTS_ROOT / "manifest.json")

Saved:
- /kaggle/working/artifacts/reports/ingestion_report.csv
- /kaggle/working/artifacts/manifest.json


# Smoke test

Проверка Definition of Done:
- загрузить FAISS
- выполнить vector search
- восстановить parent pages по page_no

In [21]:
from typing import List, Tuple

def load_global_vector_db(name: str):
    index = faiss.read_index(str(VECTOR_DIR / f"{name}.faiss"))
    meta = read_json(VECTOR_DIR / f"{name}.meta.json")
    return index, meta

def search_index(index: faiss.Index, query: str, k: int = 5) -> Tuple[List[int], List[float]]:
    if index.ntotal == 0:
        return [], []
    k_eff = min(k, int(index.ntotal))
    q_emb = embed_texts_local([query])
    scores, idxs = index.search(q_emb, k_eff)
    return idxs[0].tolist(), scores[0].tolist()

def filter_chunks_by_section_ids(chunk_meta: List[dict], idxs: List[int], section_ids: set) -> List[dict]:
    out = []
    for i in idxs:
        if not isinstance(i, int) or i < 0 or i >= len(chunk_meta):
            continue
        row = chunk_meta[i]
        if row.get("section_id") in section_ids:
            out.append(row)
    return out

query = "protection of amines and Boc chemistry"

sections_index, sections_meta = load_global_vector_db("sections")
chunks_index, chunks_meta = load_global_vector_db("chunks")

sec_idxs, sec_scores = search_index(sections_index, query, k=5)

print("Top sections:")
selected_section_ids = set()
for i, s in zip(sec_idxs, sec_scores):
    if 0 <= i < len(sections_meta):
        row = sections_meta[i]
        selected_section_ids.add(row["section_id"])
        print({
            "score": float(s),
            "doc_id": row["doc_id"],
            "section_id": row["section_id"],
            "title": row["title"],
            "pages": [row["start_page"], row["end_page"]],
        })

chunk_idxs, chunk_scores = search_index(chunks_index, query, k=100)
filtered_chunks = filter_chunks_by_section_ids(chunks_meta, chunk_idxs, selected_section_ids)

print("\nFiltered chunks:")
for row in filtered_chunks[:10]:
    print({
        "doc_id": row["doc_id"],
        "section_id": row["section_id"],
        "chunk_id": row["chunk_id"],
        "pages": [row["page_start"], row["page_end"]],
        "section_title": row["section_title"],
    })

Top sections:
{'score': 0.380950391292572, 'doc_id': '4293767636', 'section_id': '4293767636_sec_0009', 'title': '4 Для приборов класса 1 1 .1', 'pages': [19, 20]}
{'score': 0.3753091096878052, 'doc_id': 'sto_gazprom_1.1-2009', 'section_id': 'sto_gazprom_1.1-2009_sec_0003', 'title': '-в области пожарной и взрывогазовой безопасности ;', 'pages': [10, 11]}
{'score': 0.3733147978782654, 'doc_id': 'sto_gazprom_1.0-2009', 'section_id': 'sto_gazprom_1.0-2009_sec_0017', 'title': '9. ОАО « Гипроспецгаз »:', 'pages': [37, 38]}
{'score': 0.37122631072998047, 'doc_id': '4293767636', 'section_id': '4293767636_sec_0003', 'title': '6.2.21 Электродвигатели, имеющиеся в аппарате, должны соответствовать следующим требо\xad ваниям:', 'pages': [7, 8]}
{'score': 0.3639540672302246, 'doc_id': '4293767636', 'section_id': '4293767636_sec_0002', 'title': '6 Технические требования', 'pages': [5, 6]}

Filtered chunks:
{'doc_id': '4293767636', 'section_id': '4293767636_sec_0003', 'chunk_id': '4293767636_sec_0003

# Package artifacts

Формируем `artifacts.zip` для дальнейшего локального использования.

In [22]:
ZIP_PATH = Path("/kaggle/working/artifacts.zip")

def pack_artifacts(src_dir: Path, zip_path: Path) -> None:
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(
        base_name=str(zip_path).replace(".zip", ""),
        format="zip",
        root_dir=str(src_dir),
    )

pack_artifacts(ARTIFACTS_ROOT, ZIP_PATH)

print("Artifacts packed:", ZIP_PATH)
print("Size (MB):", ZIP_PATH.stat().st_size / 1024 / 1024)


Artifacts packed: /kaggle/working/artifacts.zip
Size (MB): 8.97095012664795
